# RAG from Scratch — Banking FAQ Assistant (in ~10 lines)

A minimal Retrieval-Augmented Generation pipeline, built the same way as the reference notebook, but pointed at a **banking knowledge base** and using the **free Gemini API**.

**Pipeline:** embed documents → embed query → cosine similarity search → stuff top chunks into a prompt → ask an LLM (Gemini) to answer using only that context.

Get a free Gemini API key here: https://aistudio.google.com/apikey (no credit card needed for the free tier).

#### Setup

In [1]:
!pip install -q sentence-transformers
!pip install -q google-generativeai
!pip install -q numpy

### Load the Embedding Model:

In [2]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2", device='cpu')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Build the Banking Knowledge Base:

In the reference notebook this was a Wikipedia page. Here it's a small set of bank policy / FAQ snippets — the kind of thing a retail bank's internal chatbot would be grounded on (KYC, loans, cards, payments, fraud). Swap this list for your own bank's policy docs and everything downstream stays the same.

In [3]:
paragraphs = [
"To open a savings account, a customer must complete Know Your Customer (KYC) verification by submitting a valid government photo ID (Aadhaar, Passport, or Voter ID), proof of address, and a recent passport-size photograph. Video KYC is also accepted for fully digital account opening, provided the customer completes a live video call with a bank official.",

"Re-KYC is mandatory every 2 years for high-risk customers, every 8 years for medium-risk customers, and every 10 years for low-risk customers, as per RBI guidelines. Customers who fail to complete re-KYC on time may have their account access restricted until documents are updated.",

"Home loan eligibility is generally determined by the applicant's monthly income, existing EMI obligations, credit score, and age. Most banks require a CIBIL score of 750 or above for the best interest rates, and the total EMI (including the new loan) should not exceed 40-50% of monthly income.",

"Personal loans are unsecured, meaning no collateral is required, but they typically carry higher interest rates than secured loans like home or car loans. Interest rates for personal loans usually range from 10.5% to 24% per annum depending on the applicant's credit profile and the lending bank.",

"A fixed deposit (FD) allows a customer to lock in a lump sum of money for a fixed tenure, ranging from 7 days to 10 years, in exchange for a guaranteed interest rate that is typically higher than a regular savings account. Premature withdrawal of an FD usually incurs a penalty of 0.5% to 1% on the interest rate.",

"NEFT (National Electronic Funds Transfer) settles payments in batches and can take up to two hours to reflect in the beneficiary's account. RTGS (Real Time Gross Settlement) is meant for high-value transactions above Rs 2 lakh and settles individually in real time. IMPS (Immediate Payment Service) works 24x7, including holidays, and credits the beneficiary within seconds.",

"Credit card users are entitled to an interest-free grace period, usually 20 to 50 days, on purchases if the previous month's bill is paid in full by the due date. If only the minimum amount due is paid, interest is charged on the entire outstanding balance from the date of each transaction, not just the unpaid portion.",

"Banks are required to block a lost or stolen credit or debit card immediately upon customer request through the mobile app, net banking, IVR, or by visiting a branch. Under RBI's zero-liability policy, a customer who reports unauthorized transactions within 3 working days of receiving the bank's communication bears no liability for the fraudulent transaction.",

"A CIBIL score, ranging from 300 to 900, reflects a person's creditworthiness based on their repayment history, credit utilization, length of credit history, and types of credit used. A score above 750 is generally considered good and improves the chances of loan approval at favorable interest rates.",

"Nomination facility allows an account holder to designate a person who will receive the balance in the account in the event of the account holder's death, without needing a legal heir certificate or succession certificate for amounts up to the nominee limit set by the bank.",

"Minimum balance requirements vary by account type: regular savings accounts often require Rs 1,000 to Rs 10,000 depending on whether the branch is in a rural, semi-urban, or metro area. Basic Savings Bank Deposit Accounts (BSBDA), meant for financial inclusion, have no minimum balance requirement.",

"Banks use two-factor authentication (2FA) for online transactions, typically combining something the customer knows (PIN or password) with something they have (an OTP sent to a registered mobile number). Customers are advised to never share OTPs, as banks never ask for them over phone calls or email.",

"A loan against a fixed deposit lets customers borrow up to 90-95% of the FD value, usually at an interest rate just 1-2% above the FD's own interest rate, while the deposit continues to earn interest. This is generally cheaper and faster to obtain than a personal loan.",

"Cheque payments in India follow the CTS (Cheque Truncation System), which processes an electronic image of the cheque instead of moving the physical instrument between banks, reducing clearing time to within one working day for most cases.",

"If a customer disputes a transaction, banks are required to resolve the complaint within specific turnaround times set by the RBI: unauthorized electronic transactions must generally be resolved within 90 days, while failed ATM transactions must be reversed within 5 working days, with a penalty payable to the customer for delays beyond that.",
]

In [4]:
import textwrap

for i, p in enumerate(paragraphs):
  wrapped_text = textwrap.fill(p, width=100)

  print("-----------------------------------------------------------------")
  print(f"[{i}]")
  print(wrapped_text)
  print("-----------------------------------------------------------------")

-----------------------------------------------------------------
[0]
To open a savings account, a customer must complete Know Your Customer (KYC) verification by
submitting a valid government photo ID (Aadhaar, Passport, or Voter ID), proof of address, and a
recent passport-size photograph. Video KYC is also accepted for fully digital account opening,
provided the customer completes a live video call with a bank official.
-----------------------------------------------------------------
-----------------------------------------------------------------
[1]
Re-KYC is mandatory every 2 years for high-risk customers, every 8 years for medium-risk customers,
and every 10 years for low-risk customers, as per RBI guidelines. Customers who fail to complete re-
KYC on time may have their account access restricted until documents are updated.
-----------------------------------------------------------------
-----------------------------------------------------------------
[2]
Home loan eligibil

### Embed the Documents:

In [5]:
docs_embed = model.encode(paragraphs, normalize_embeddings=True)

In [6]:
docs_embed.shape

(15, 384)

### Embed the Query:

In [7]:
query = "What documents do I need for KYC, and how often do I have to redo it?"
query_embed = model.encode(query, normalize_embeddings=True)

In [8]:
query_embed.shape

(384,)

### Find the Closest Paragraphs to the Query:

In [9]:
import numpy as np
similarities = np.dot(docs_embed, query_embed.T)

In [10]:
similarities

array([0.46260583, 0.613237  , 0.17246678, 0.0616702 , 0.10626682,
       0.07276003, 0.04341889, 0.04549405, 0.08863796, 0.1533868 ,
       0.13659652, 0.04995916, 0.01475782, 0.07374838, 0.0597304 ],
      dtype=float32)

In [11]:
top_3_idx = np.argsort(similarities, axis=0)[-3:][::-1].tolist()
top_3_idx

[1, 0, 2]

In [12]:
most_similar_documents = [paragraphs[idx] for idx in top_3_idx]

In [13]:
CONTEXT = ""
for i, p in enumerate(most_similar_documents):
  wrapped_text = textwrap.fill(p, width=100)

  print("-----------------------------------------------------------------")
  print(wrapped_text)
  print("-----------------------------------------------------------------")
  CONTEXT += wrapped_text + "\n\n"

-----------------------------------------------------------------
Re-KYC is mandatory every 2 years for high-risk customers, every 8 years for medium-risk customers,
and every 10 years for low-risk customers, as per RBI guidelines. Customers who fail to complete re-
KYC on time may have their account access restricted until documents are updated.
-----------------------------------------------------------------
-----------------------------------------------------------------
To open a savings account, a customer must complete Know Your Customer (KYC) verification by
submitting a valid government photo ID (Aadhaar, Passport, or Voter ID), proof of address, and a
recent passport-size photograph. Video KYC is also accepted for fully digital account opening,
provided the customer completes a live video call with a bank official.
-----------------------------------------------------------------
-----------------------------------------------------------------
Home loan eligibility is gener

### Build the Prompt:

In [14]:
prompt = f"""
You are a helpful assistant for a retail bank's customer support desk.
Use the following CONTEXT to answer the QUESTION at the end.
If the answer isn't in the CONTEXT, just say you don't know — don't make anything up, and don't give financial advice beyond what's stated.

CONTEXT: {CONTEXT}
QUESTION: {query}
"""
print(prompt)


You are a helpful assistant for a retail bank's customer support desk.
Use the following CONTEXT to answer the QUESTION at the end.
If the answer isn't in the CONTEXT, just say you don't know — don't make anything up, and don't give financial advice beyond what's stated.

CONTEXT: Re-KYC is mandatory every 2 years for high-risk customers, every 8 years for medium-risk customers,
and every 10 years for low-risk customers, as per RBI guidelines. Customers who fail to complete re-
KYC on time may have their account access restricted until documents are updated.

To open a savings account, a customer must complete Know Your Customer (KYC) verification by
submitting a valid government photo ID (Aadhaar, Passport, or Voter ID), proof of address, and a
recent passport-size photograph. Video KYC is also accepted for fully digital account opening,
provided the customer completes a live video call with a bank official.

Home loan eligibility is generally determined by the applicant's monthly in

### Call Gemini (Free API):

Grab a free key from [Google AI Studio](https://aistudio.google.com/apikey). If you're in Colab, the cleanest way is to store it as a Colab secret named `GEMINI_API_KEY` (key icon in the left sidebar) and read it with `userdata.get`. Locally, just set it as an environment variable or paste it in directly.

In [18]:
import os
import google.generativeai as genai

# Google Colab secret
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel("gemini-2.5-flash")  # fast + free-tier friendly; swap if this model is retired

In [16]:
response = gemini.generate_content(prompt)
print(response.text)

To complete KYC, you need to submit a valid government photo ID (Aadhaar, Passport, or Voter ID), proof of address, and a recent passport-size photograph. Video KYC is also accepted.

How often you have to redo KYC (Re-KYC) depends on your customer risk category:
*   High-risk customers: Every 2 years
*   Medium-risk customers: Every 8 years
*   Low-risk customers: Every 10 years


### Try Your Own Question:

Re-run retrieval + generation for any new banking question, end-to-end, in one cell.

In [17]:
def ask_bank_bot(question, k=3):
    q_embed = model.encode(question, normalize_embeddings=True)
    sims = np.dot(docs_embed, q_embed.T)
    top_idx = np.argsort(sims, axis=0)[-k:][::-1].tolist()
    context = "\n\n".join(paragraphs[i] for i in top_idx)

    prompt = f"""Use the following CONTEXT to answer the QUESTION.
If you don't know, say so — don't make anything up.

CONTEXT: {context}
QUESTION: {question}
"""
    return gemini.generate_content(prompt).text

print(ask_bank_bot("If my debit card gets stolen, am I liable for fraudulent charges?"))

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1748.76ms


If you report unauthorized transactions within 3 working days of receiving the bank's communication, you bear no liability for the fraudulent transaction, under RBI's zero-liability policy.


---
### Next steps to make this production-grade

- **Chunking**: real bank policy PDFs need smarter splitting (by section/heading, with overlap) instead of one-liners.
- **Vector store**: swap the in-memory `numpy` dot product for FAISS / Chroma / pgvector once the doc set grows past a few hundred chunks.
- **Guardrails**: add a check that refuses to answer questions needing personal account data (balance, transaction history) from this kind of general-FAQ RAG — that needs an authenticated, per-customer data path instead.
- **Evaluation**: track retrieval hit-rate and answer faithfulness (e.g. with a small labeled set of Q&A pairs) before shipping.